# VECTORBT

In [ ]:
import importlib
import src.utils.db as db_module
import src.utils.store as store_module
import src.utils.theme as theme_module

importlib.reload(db_module)
importlib.reload(store_module)
importlib.reload(theme_module)

from pathlib import Path
from typing import cast
from src.utils.db import Db
import vectorbt as vbt
from src.utils.store import Store
from src.utils.theme import vscode_dark
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import os

db = Db("vectorbt/tutorial")

In [12]:
PYTHONPATH = os.environ.get("PYTHONPATH").split(":")[0]
ARTIFACTS_RELPATH = Path("artifacts/charts")
ARTIFACTS_ABSPATH = PYTHONPATH / ARTIFACTS_RELPATH
dfs = Store[pd.DataFrame]()
inds = Store[vbt.indicators.IndicatorBase]()
signals = Store[any]()
pfs = Store[vbt.Portfolio]()
trs = Store[any]()
views = Store[any]()

In [13]:
end_time = datetime.now()
start_time = end_time - timedelta(days=1)

dfs.latest = "BTC-USD-Close", pd.DataFrame(
    vbt.YFData.download(
        ["BTC-USD", "ETH-USD"],
        missing_index="drop",
        start=start_time,
        end=end_time,
        interval="1m",
    ).get("Close")
)
dfs.latest

/opt/conda/envs/econ/lib/python3.13/site-packages/vectorbt/data/base.py:527: UserWarning:

Symbols have mismatching index. Dropping missing data points.



symbol,BTC-USD,ETH-USD
Datetime,,
2025-05-11 18:12:00+00:00,103964.609375,2484.297119
2025-05-11 18:14:00+00:00,103958.953125,2483.925293
2025-05-11 18:16:00+00:00,103931.671875,2481.769043
2025-05-11 18:19:00+00:00,103987.148438,2485.308838
2025-05-11 18:22:00+00:00,104040.687500,2487.375488
...,...,...
2025-05-12 18:03:00+00:00,102606.476562,2480.038574
2025-05-12 18:04:00+00:00,102618.109375,2484.194336
2025-05-12 18:05:00+00:00,102615.539062,2483.718262


In [14]:
def rsi5_ma_indicator(
    close_pd: pd.Series,
    rsi_window: int,
    rsi_low: int,
    rsi_high: int,
    ma_window: int,
):
    close_5m = close_pd.resample("5min").last()
    rsi_5m = vbt.RSI.run(close_5m, window=rsi_window).rsi
    rsi_pd = rsi_5m.align(close_pd, join="right", axis=0)[0].ffill()
    ma = vbt.MA.run(close_pd, ma_window).ma.to_numpy()
    close = close_pd.to_numpy()
    rsi = rsi_pd.to_numpy()
    trend = np.where(rsi > rsi_high, -1, 0)
    trend = np.where((rsi < rsi_low) & (close < ma), 1, trend)
    return trend


inds.latest = "RSI5_MA", vbt.IndicatorFactory(
    class_name="RSI5_MA",
    short_name="R5M",
    input_names=["close"],
    param_names=["rsi_window", "rsi_low", "rsi_high", "ma_window"],
    output_names=["values"],
).from_apply_func(
    rsi5_ma_indicator,
    rsi_window=14,
    rsi_low=30,
    rsi_high=70,
    ma_window=50,
    keep_pd=True,
)

In [15]:
signals.latest = (
    "RSI5_MA",
    inds.latest.run(
        dfs.latest,
        rsi_window=14,
        rsi_low=30,
        rsi_high=70,
        ma_window=50,
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RSI5_MA", vbt.Portfolio.from_signals(dfs.latest, entries, exits)
pfs.latest[pfs.latest.total_return().idxmax()].stats()

/opt/conda/envs/econ/lib/python3.13/site-packages/vectorbt/generic/stats_builder.py:396: UserWarning:

Metric 'sharpe_ratio' requires frequency to be set

/opt/conda/envs/econ/lib/python3.13/site-packages/vectorbt/generic/stats_builder.py:396: UserWarning:

Metric 'calmar_ratio' requires frequency to be set

/opt/conda/envs/econ/lib/python3.13/site-packages/vectorbt/generic/stats_builder.py:396: UserWarning:

Metric 'omega_ratio' requires frequency to be set

/opt/conda/envs/econ/lib/python3.13/site-packages/vectorbt/generic/stats_builder.py:396: UserWarning:

Metric 'sortino_ratio' requires frequency to be set



Start                         2025-05-11 18:12:00+00:00
End                           2025-05-12 18:08:00+00:00
Period                                              987
Start Value                                       100.0
End Value                                     99.536469
Total Return [%]                              -0.463531
Benchmark Return [%]                          -1.034339
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                               3.912328
Max Drawdown Duration                             276.0
Total Trades                                          3
Total Closed Trades                                   2
Total Open Trades                                     1
Open Trade PnL                                -3.251066
Win Rate [%]                                      100.0
Best Trade [%]                                 2.748825
Worst Trade [%]                                0

In [16]:
pfs.latest

## Volume

In [ ]:
name = "RM_volume"
signals.latest = (
    name,
    inds.latest.run(
        dfs.latest,
        param_product=True,
        rsi_window=np.arange(10, 40, step=3, dtype=int),
        # ma_window=np.arange(10, 100, step=14, dtype=int),
        rsi_low=np.arange(10, 40, step=3, dtype=int),
        rsi_high=np.arange(60, 80, step=3, dtype=int),
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = name, vbt.Portfolio.from_signals(dfs.latest, entries, exits)
trs.latest = name, pfs.latest.total_return()
views.latest = (
    name,
    trs.latest.groupby(level=["R5M_rsi_low", "R5M_rsi_high", "symbol"]).mean(),
)
trs.latest.vbt.volume(
    x_level="R5M_rsi_low",
    y_level="R5M_rsi_high",
    z_level="R5M_rsi_window",
    slider_level="symbol",
).write_html(ARTIFACTS_ABSPATH / "volume2.html")